# 🚀 OCR TownHub — CHỈ CHẠY SERVICE (đã train sẵn)

Dùng khi **đã train** và weight đã nằm trong `Google Drive/townhub_ocr/`.
Không sinh dataset, không train. **Runtime → Run all** (nhớ Restart sau bước 1).

Cần trên Drive: `townhub_ocr/weights/vietocr_invoice.pth`, `townhub_ocr/inference/rec_vi`, `.../det_vi`, `townhub_ocr/dict_vi.txt`.


### 1 — Cài thư viện *(chạy 1 lần → Restart session)*
> Cảnh báo đỏ về numpy của jax/shap… là vô hại, bỏ qua.


In [ ]:
!nvidia-smi -L || echo '⚠️ Chưa bật GPU: Runtime → Change runtime type → T4 GPU'
!pip -q install fastapi uvicorn pydantic requests pdf2image pillow google-generativeai
!pip -q install torch easyocr vietocr paddlepaddle-gpu==2.6.1 paddleocr==2.7.3
# Ép NumPy 1.x cho khớp ABI của cv2/paddle (đây là bước xử lý lỗi 'numpy.core.multiarray failed to import').
!pip -q install 'numpy==1.26.4' 'opencv-python-headless==4.9.0.80'
print('✅ Cài xong. BÂY GIỜ: Runtime → Restart session, rồi chạy tiếp A.2 (KHÔNG chạy lại A.1).')


### 2 — Cấu hình + trỏ weight trên Drive


In [ ]:
# Chạy MỖI phiên (và mỗi lần sau khi Restart). Thiết lập thư mục + đường dẫn weight trên Drive.
import os
REPO_URL = 'https://github.com/thuongerikdev/TownHub'
if not os.path.exists('/content/townhub'):
    !git clone --depth 1 $REPO_URL /content/townhub
os.chdir('/content/townhub/ocr-service'); print('cwd:', os.getcwd())

from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/townhub_ocr'
os.makedirs(DRIVE, exist_ok=True)

# Service đọc weight qua các biến môi trường này (trỏ thẳng vào Drive).
os.environ['VIETOCR_WEIGHTS'] = f'{DRIVE}/weights/vietocr_invoice.pth'
os.environ['PADDLE_REC_DIR']  = f'{DRIVE}/inference/rec_vi'
os.environ['PADDLE_DET_DIR']  = f'{DRIVE}/inference/det_vi'
os.environ['PADDLE_REC_DICT'] = f'{DRIVE}/dict_vi.txt'
os.environ['GEMINIKEY'] = 'DAN_KEY_GEMINI_CUA_BAN'   # chỉ cần nếu dùng engine gemini
os.environ['OCRKEY']    = 'doan-ocr-2026'            # khớp OCR_API_KEY phía .NET

# Đã có weight trên Drive chưa? -> quyết định train hay chạy thẳng.
_have = all(os.path.exists(os.environ[k]) for k in ['VIETOCR_WEIGHTS','PADDLE_REC_DIR','PADDLE_DET_DIR'])
print('✅ ĐÃ có weight trên Drive → có thể BỎ QUA phần B, sang thẳng phần C (CHẠY).' if _have
      else 'ℹ️ CHƯA có weight → chạy phần B (TRAINING) trước.')


### 3 — Kiểm tra weight


In [ ]:
# Kiểm tra weight đã sẵn trên Drive chưa trước khi chạy service.
import os
for k in ['VIETOCR_WEIGHTS','PADDLE_REC_DIR','PADDLE_DET_DIR','PADDLE_REC_DICT']:
    ok = os.path.exists(os.environ[k])
    print(('✅' if ok else '❌ THIẾU'), k, '=', os.environ[k])
print('\nCó ❌ nghĩa là chưa train (chạy phần B) hoặc chưa lưu lên Drive.')


### 4 — Mở tunnel cloudflared


In [ ]:
# Mở tunnel cloudflared -> URL https công khai để backend .NET gọi vào (đặt vào OCR_SERVICE_URL).
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
import subprocess, re
p = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:7860','--no-autoupdate'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m: print('\n🌐 OCR_SERVICE_URL =', m.group(0)); break


### 5 — Chạy service *(giữ chạy)*


In [ ]:
!python app.py
